# H&M baseline: bảng feature theo khách hàng
Chạy `00_setup_colab.ipynb` để cài package, sau đó chạy notebook này. Kết quả chỉ có `customer_id` và 384 feature: 128 transaction, 128 text, 128 image. Không có nhãn churn hoặc dự đoán 0/1.

## 1. Mount Drive
Đăng nhập tài khoản chứa ZIP H&M. Không tải hoặc giải nén ZIP vào repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Chọn dữ liệu và mốc quan sát
`OBSERVATION_END` dưới đây chỉ dùng để chạy thử pipeline; luận văn chưa chốt observation/prediction windows. `MAX_CUSTOMERS=16` là mẫu kỹ thuật, không dùng để báo cáo kết quả nghiên cứu.

In [ ]:
from pathlib import Path
from multimodal_churn.utils.config import load_config, resolve_dataset_paths

DATASETS_ROOT = Path('/content/drive/MyDrive/Project Master Thesis/Data/Datasets')
ARTIFACTS_ROOT = Path('/content/drive/MyDrive/multimodal-churn-fusion')
OBSERVATION_END = '2020-08-31'  # Demo only; not a finalized churn cutoff.
OBSERVATION_START = None
MAX_CUSTOMERS = 16

baseline_path = Path('configs/baseline.yaml')
config = load_config(baseline_path, dataset_path=Path('configs/datasets/hm.yaml'))
paths = resolve_dataset_paths(config, DATASETS_ROOT)
archive_path = paths['archive']
if not archive_path.is_file():
    raise FileNotFoundError(f'Check the mounted Drive account and path: {archive_path}')
output_dir = ARTIFACTS_ROOT / 'features' / f'hm_demo_{OBSERVATION_END}'
print('Archive:', archive_path)
print('Output:', output_dir)

## 3. Tạo bảng feature
Lần chạy đầu sẽ tải trọng số MiniLM và CLIP công khai. Transformer giao dịch và phép chiếu 128D chưa được huấn luyện; bảng này kiểm tra luồng vector và phép concat, chưa chứng minh khả năng dự đoán.

In [ ]:
from multimodal_churn.feature_pipeline import build_hm_feature_table

table = build_hm_feature_table(
    config=config,
    archive_path=archive_path,
    observation_start=OBSERVATION_START,
    observation_end=OBSERVATION_END,
    max_customers=MAX_CUSTOMERS,
    output_dir=output_dir,
)
print('Rows, columns:', table.shape)
print('CSV:', output_dir / 'features.csv')

## 4. Kiểm tra bảng
Một dòng là một khách hàng. Thứ tự cột là transaction → text → image, mỗi khối 128 chiều. File `manifest.json` ghi mốc quan sát, độ phủ modality và trạng thái chưa huấn luyện.

In [ ]:
import json
assert table.shape[1] == 1 + 3 * config['model']['common_embedding_dim']
assert 'label' not in table.columns and 'churn' not in table.columns
print(json.loads((output_dir / 'manifest.json').read_text(encoding='utf-8')))
display(table[['customer_id', 'transaction_000', 'text_000', 'image_000']].head())